# Bài 3 · Đánh giá phân loại · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: số phiếu spam v (0–11) của kNN Bài 2 v3 cho 1.032 tin validation và 1.032 tin test. Spam là lớp dương; ma trận theo nhãn [0, 1].
Các tin nhắn chỉ dùng để học: không gọi số hoặc truy cập liên kết trong nội dung. Không chọn ngưỡng bằng test.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
ROOT = Path.cwd()
val = pd.read_csv(ROOT / 'data/sms_validation_votes_v3.csv')
test = pd.read_csv(ROOT / 'data/sms_test_votes_v3.csv')
locked = pd.read_csv(ROOT / 'data/sms_test_predictions_v3.csv')
print(val.columns.tolist())
print('validation:', len(val), 'tin,', val.y.sum(), 'spam · test:', len(test), 'tin,', test.y.sum(), 'spam')
# Tệp số phiếu test khớp tệp dự đoán đã khoá của Bài 2 v3
assert (test.set_index('message_id').loc[locked.message_id, 'prediction'].values == locked.prediction.values).all()

['message_id', 'source_record', 'label', 'y', 'n_chars', 'n_digits', 'digit_fraction', 'votes', 'prediction']
validation: 1032 tin, 129 spam · test: 1032 tin, 128 spam


## Chặng 1 · Đúng 97% đã đủ tốt?

In [2]:
# Mốc luôn ham và kNN (v ≥ 6) trên test
print('luôn ham: accuracy', round((test.y == 0).mean(), 4), '· spam bắt được 0 /', test.y.sum())
p6 = (test.votes >= 6).astype(int)
cm = confusion_matrix(test.y, p6, labels=[0, 1])
print(cm)                      # [[TN, FP], [FN, TP]]
tn, fp, fn, tp = cm.ravel()
print('lề hàng:', tn + fp, fn + tp, '· lề cột:', tn + fn, fp + tp, '· accuracy', round((tn + tp) / len(test), 4))

luôn ham: accuracy 0.876 · spam bắt được 0 / 128
[[891  13]
 [ 17 111]]
lề hàng: 904 128 · lề cột: 908 124 · accuracy 0.9709


In [3]:
# Hai tin minh hoạ của slide: bản ghi 4552 (FP) và 1501 (FN)
test[test.source_record.isin([4552, 1501])][['source_record', 'y', 'votes']]

,source_record,y,votes
43,4552,0,6
555,1501,1,4


## Chặng 2 · Precision và recall: hai mẫu số

In [4]:
P = tp / (tp + fp); R = tp / (tp + fn); F1 = 2 * tp / (2 * tp + fp + fn)
print(f'P = {tp}/{tp + fp} = {P:.4f} · R = {tp}/{tp + fn} = {R:.4f} · F1 = {2 * tp}/{2 * tp + fp + fn} = {F1:.4f}')
print('scikit-learn:', round(precision_score(test.y, p6), 4), round(recall_score(test.y, p6), 4), round(f1_score(test.y, p6), 4))

P = 111/124 = 0.8952 · R = 111/128 = 0.8672 · F1 = 222/252 = 0.8810
scikit-learn: 0.8952 0.8672 0.881


In [5]:
# Mười tin mô phỏng: tự kiểm phép tính tay
toy = pd.DataFrame({'id': list('ABCDEFGHIJ'), 'v': [11, 10, 9, 8, 7, 6, 4, 3, 2, 1], 'y': [1, 0, 1, 1, 0, 1, 0, 0, 0, 0]})
for m in (6, 9, 10):
    p = (toy.v >= m).astype(int)
    print('m =', m, confusion_matrix(toy.y, p, labels=[0, 1]).ravel(), '· P =', round(precision_score(toy.y, p), 3), '· R =', recall_score(toy.y, p))

m = 6 [4 2 0 4] · P = 0.667 · R = 1.0
m = 9 [5 1 2 2] · P = 0.667 · R = 0.5
m = 10 [5 1 3 1] · P = 0.5 · R = 0.25


In [6]:
# Đoạn mã của slide 51, chạy trên test với m = 6
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score
y, v, m = test.y.values, test.votes.values, 6
y_du_doan = (v >= m).astype(int)               # quy tắc v ≥ m
tn, fp, fn, tp = confusion_matrix(y, y_du_doan, labels=[0, 1]).ravel()
precision = tp / (tp + fp) if tp + fp else 0.0  # ghi rõ quy ước khi không có cảnh báo
recall    = tp / (tp + fn)
f1        = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
assert np.isclose(precision, precision_score(y, y_du_doan, zero_division=0))
print(tn, fp, fn, tp, round(precision, 4), round(recall, 4), round(f1, 4))

891 13 17 111 0.8952 0.8672 0.881


## Chặng 3 · Ngưỡng: cùng điểm, khác quyết định (chỉ validation)

In [7]:
rows = []
for m in range(1, 13):
    p = (val.votes >= m).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(val.y, p, labels=[0, 1]).ravel()
    rows.append(dict(m=m, FP=fp_, FN=fn_, TP=tp_, canh_bao=tp_ + fp_, precision=precision_score(val.y, p, zero_division=0), recall=recall_score(val.y, p)))
bang = pd.DataFrame(rows).round(4)
bang

,m,FP,FN,TP,canh_bao,precision,recall
0,1,144,8,121,265,0.4566,0.9380
1,2,33,10,119,152,0.7829,0.9225
2,3,27,11,118,145,0.8138,0.9147
3,4,18,12,117,135,0.8667,0.9070
4,5,14,14,115,129,0.8915,0.8915
5,6,9,16,113,122,0.9262,0.8760
6,7,6,18,111,117,0.9487,0.8605
7,8,4,19,110,114,0.9649,0.8527
8,9,4,24,105,109,0.9633,0.8140
9,10,3,35,94,97,0.9691,0.7287


Câu hỏi: (1) Cột nào không bao giờ tăng khi nâng m? (2) Tìm một chỗ precision giảm khi nâng m. (3) m = 12: vì sao precision ở đây là 0? Đó là quy ước `zero_division=0`, không phải precision quan sát được.

## Chặng 4 · Chọn ngưỡng theo chi phí (validation), áp lên test một lần

In [8]:
def chon(c_fp, c_fn):
    b = bang.assign(chi_phi=c_fp * bang.FP + c_fn * bang.FN).sort_values(['chi_phi', 'canh_bao', 'm'], ascending=[True, True, False])
    return int(b.m.iloc[0]), int(b.chi_phi.iloc[0])
for kb in [(5, 1), (1, 5), (3, 1), (10, 1)]:
    print('c_FP, c_FN =', kb, '→ (m, chi phí) =', chon(*kb))

c_FP, c_FN = (5, 1) → (m, chi phí) = (8, 39)
c_FP, c_FN = (1, 5) → (m, chi phí) = (4, 78)
c_FP, c_FN = (3, 1) → (m, chi phí) = (8, 31)
c_FP, c_FN = (10, 1) → (m, chi phí) = (8, 59)


In [9]:
# Khoá m theo 5·FP + FN, áp lên test (test lịch sử: đã chấm ở Bài 2)
m_khoa = chon(5, 1)[0]
pt = (test.votes >= m_khoa).astype(int)
print('m =', m_khoa, '· TN, FP, FN, TP =', confusion_matrix(test.y, pt, labels=[0, 1]).ravel())

m = 8 · TN, FP, FN, TP = [897   7  21 107]


Viết 100–150 từ: ngưỡng đã chọn tốt hơn m = 6 ở tiêu chí nào, kém hơn ở đâu, và ba giới hạn của con số test này.